# Week 8 Listing Summarization Demo

This notebook shows only:

1. ROUGE-L pass/fail result
2. 20 listings with their corresponding generated summary


In [18]:
from __future__ import annotations

import csv
import json
import re
import sys
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.listing_summarizer import ListingSummarizer
from scripts.validate_week8_summarization import run_validation

In [19]:
summarizer = ListingSummarizer()
metrics = run_validation()

rouge_l = metrics["rouge_l"]
threshold = metrics["rouge_threshold"]
passed = rouge_l > threshold

print(f"ROUGE-L: {rouge_l:.4f}")
print(f"Threshold: {threshold:.2f}")
print(f"Status: {'PASS' if passed else 'FAIL'}")

ROUGE-L: 0.7252
Threshold: 0.40
Status: PASS


In [20]:
def parse_location(text: str) -> str:
    direct_match = re.search(
        r"\b(?:in|near|around|within)\s+([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){0,4})",
        text,
    )
    if direct_match:
        location = direct_match.group(1).strip()
        if location.lower() not in {"the", "this", "a", "an"}:
            return location

    community_match = re.search(
        r"\b([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){0,4})\s+(Community|Neighborhood|District|Village)\b",
        text,
    )
    if community_match:
        return f"{community_match.group(1).strip()} {community_match.group(2).strip()}"

    downtown_match = re.search(
        r"\bdowntown\s+([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){0,3})",
        text,
        flags=re.I,
    )
    if downtown_match:
        return f"Downtown {downtown_match.group(1).strip()}"

    cap_phrase = re.search(r"\b([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){1,4})\b", text)
    if cap_phrase:
        return cap_phrase.group(1).strip()

    return "Local area"


def listing_from_record(record: dict) -> dict:
    beds = None
    baths = None
    price = None
    amenities: list[str] = []
    for entity in record.get("entities", []):
        label = entity.get("label")
        value = entity.get("value")
        if label == "BEDROOMS" and beds is None:
            beds = value
        elif label == "BATHROOMS" and baths is None:
            baths = value
        elif label == "PRICE" and price is None and isinstance(value, (int, float)) and value >= 50000:
            price = value
        elif label == "AMENITY" and isinstance(value, str):
            cleaned = value.strip().lower()
            if cleaned and cleaned not in amenities:
                amenities.append(cleaned)

    cleaned_remarks = re.sub(r"\s+", " ", record.get("text", "")).strip()

    return {
        "listing_id": record.get("id", "unknown"),
        "city": parse_location(record.get("text", "")),
        "bedrooms": beds if beds is not None else 3,
        "bathrooms": baths if baths is not None else 2,
        "price": price if price is not None else 750000,
        "features": amenities[:4],
        "remarks": cleaned_remarks,
    }


records_path = PROJECT_ROOT / "data" / "processed" / "remarks_labeled.jsonl"
records: list[dict] = []
with records_path.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

listings = [listing_from_record(record) for record in records[:20]]
len(listings)

20

In [21]:
summary_rows = [
    {
        "listing_id": listing["listing_id"],
        "actual_listing": listing["remarks"],
        "listing_summary": summarizer.extractive_summary(listing, num_sentences=2),
        "rating_factuality_1_to_5": "",
        "rating_coverage_1_to_5": "",
        "rating_fluency_1_to_5": "",
        "annotator_notes": "",
    }
    for listing in listings
]

annotation_csv = PROJECT_ROOT / "data" / "processed" / "week8_human_annotation_sheet.csv"
with annotation_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "listing_id",
            "actual_listing",
            "listing_summary",
            "rating_factuality_1_to_5",
            "rating_coverage_1_to_5",
            "rating_fluency_1_to_5",
            "annotator_notes",
        ],
    )
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"Total listings shown: {len(summary_rows)}")
print(f"CSV updated: {annotation_csv}")

if pd is not None:
    display(pd.DataFrame(summary_rows)[["listing_id", "actual_listing", "listing_summary"]])
else:
    for i, row in enumerate(summary_rows, start=1):
        print(f"{i:02d}. listing_id={row['listing_id']}")
        print(f"    actual_listing: {row['actual_listing']}")
        print(f"    listing_summary: {row['listing_summary']}")
        print()

Total listings shown: 20
CSV updated: /Users/nathanye/Documents/GitHub/nlp-internship/data/processed/week8_human_annotation_sheet.csv


,listing_id,actual_listing,listing_summary
0,1115056072,"Stunning, luxury duplex located close to world...","2 bed, 2 bath home in Downtown San Clemente li..."
1,1145958098,Beautiful Dorado Model on the Golf CourseExper...,"2 bed, 2.5 bath home in Beautiful Dorado Model..."
2,1144563210,Beautiful Duplex for sale! Great for 1st time ...,"2 bed, 1 bath home in Beautiful Duplex listed ..."
3,1130265863,"Tucked away on over 6.7 acres of scenic, usabl...","3 bed, 2 bath home in Northern California list..."
4,1139661999,"Located in the Flores Montanas community, this...","4 bed, 5 bath home in Flores Montanas listed a..."
5,1131854913,Your Mountain Retreat Awaits. Dreaming of a pe...,"3 bed, 2 bath home in Lake Gregory listed at $..."
6,1151361631,Located within the highly sought-after Temple ...,"2 bed, 2 bath home in Temple City School Distr..."
7,1147142692,Thoughtfully designed to embrace its breathtak...,"3 bed, 2 bath home in BBQ listed at $750,000 w..."
8,1151836006,Step inside this beautifully upgraded 2-bedroo...,"3 bed, 2 bath home in La Costa listed at $750,..."
9,1150619006,"Fully renovated and move-in ready, this pool-f...","3 bed, 2 bath home in Trader Joe listed at $75..."


In [5]:
# intentionally left empty

Run cells top-to-bottom to see:

- ROUGE-L pass/fail
- 20 listings with corresponding summaries